# PyroGuard AI 🔥🇮🇩: Discord Bot Operational Manual
### Interactive Slash Command Reference & Technical Verification Notebook

This Jupyter Notebook provides the official interactive instructions and operational guide for the **PyroGuard AI Discord Bot**. It covers Discord Developer Portal setup, slash command specifications (`/risk`, `/hotspots`, `/ask`, `/regions`), underlying remote sensing algorithms, and executable Python demonstrations simulating bot command responses.

## 1. Prerequisites & Discord Developer Portal Setup

To connect the Discord Bot to your Discord server, follow these steps:

1. **Create an Application**:
   - Visit the [Discord Developer Portal](https://discord.com/developers/applications) and create a **New Application** named `PyroGuard AI`.
2. **Obtain the Bot Token**:
   - Navigate to **Settings → Bot**.
   - Under **Build-A-Bot**, click **Reset Token** and copy the resulting string (`~70` characters, e.g. `MTIz...G1x...abc...`).
   - Enable **Message Content Intent** under Privileged Gateway Intents.
3. **Generate OAuth2 Invite URL**:
   - Navigate to **OAuth2 → URL Generator**.
   - In **Scopes**, check: `bot` and `applications.commands`.
   - In **Bot Permissions**, check: `Send Messages`, `Embed Links`, `Attach Files`, `Read Message History`.
   - Open the generated invite link in your browser and authorize the bot into your incident channel.
4. **Configure Environment**:
   - Add the copied token to your `.env` file:
   ```env
   DISCORD_BOT_TOKEN=your_token_here
   ```
5. **Start the Bot**:
   ```bash
   ./run_discord_bot.sh
   ```

## 2. Command Architecture & Quick Reference Table

| Slash Command | Arguments | Primary Remote Sensing Engine | Output Returned |
| :--- | :--- | :--- | :--- |
| **`/risk`** | `region` (autocomplete) | Sentinel-1 SAR (VV/VH), Sentinel-2 NDWI, ONNX Model | WVI Score (0.0–1.0), Peat Water Table depth / Savanna classification, Top Drivers, Advisory |
| **`/hotspots`** | `region`, `days` (default: 3) | NASA FIRMS VIIRS 375m NRT + KLHK KHG layer | Active thermal detections, Fire Radiative Power (MW), Peat overlap, Smoke Plumes |
| **`/ask`** | `query` (free text) | Autonomous ReAct Agent Loop | Multi-tool reasoning steps, situational briefing, and tactical response report |
| **`/regions`** | *None* | Geographic Registry | Complete directory of monitored Indonesian provinces, peat domes, and national parks |

---
## 3. Command: `/risk [region]`

### Technical Description
The `/risk` command calculates the **Wildfire Vulnerability Index (WVI)** across a 14-day forecast window. It evaluates coupled surface desiccation using:
- **Sentinel-1 SAR C-band (VV backscatter drop $\Delta\sigma^0_{\text{VV}}$)**: Detects moisture drawdown in tropical peat domes (*kubah gambut*) and highland topsoils.
- **Peatland Drying Index ($I_{\text{PDI}}$)**: In peatland zones, estimates water table depth relative to the Indonesian Peat Restoration Agency (BRGM) statutory danger threshold of **-40 cm**:
  $$I_{\text{PDI}} = 0.50 \cdot f_{\text{SAR}}(\Delta\sigma^0_{\text{VV}}) + 0.25 \cdot f_{\text{LST}}(\Delta T_s) + 0.25 \cdot \exp\left(-\frac{R_{14}}{25}\right)$$
- **Ecosystem Adaptation**: Automatically distinguishes **Tropical Peatlands** (Kalimantan/Sumatra) from **Highland Volcanic Savannas** (such as **Mount Bromo National Park** in East Java), modifying tactical recommendations accordingly.

### Risk Tier Classification Matrix
- 🔴 **CRITICAL** (≥ 0.80): Red Embed (`#DC2626`) — High ignition probability, water table critically depleted.
- 🟠 **HIGH** (0.60 – 0.79): Orange Embed (`#EA580C`) — Rapid drying, intense ground patrols required.
- 🟡 **MODERATE** (0.35 – 0.59): Yellow Embed (`#CA8A04`) — Moisture stress in agricultural concessions.
- 🟢 **LOW** (< 0.35): Green Embed (`#16A34A`) — Normal moisture baseline.

In [ ]:
# Executable Demonstration: Simulate /risk Command for Mount Bromo and Kapuas Regency
import os
import sys
sys.path.insert(0, os.path.abspath('backend'))

import asyncio
from app.agent.tools import tool_extract_regional_fire_metrics, tool_forecast_wildfire_risk
from app.domain.indonesia import resolve_region

async def test_risk_command(region_query: str):
    reg = resolve_region(region_query)
    print(f'\n{"="*60}')
    print(f"Executing /risk region: '{reg['name']}' ({reg['province']})")
    print(f"Ecosystem: {reg.get('ecosystem_type', 'Tropical Forest')}")
    
    metrics = await tool_extract_regional_fire_metrics(region_name=reg['name'], lookback_days=30)
    forecast = await tool_forecast_wildfire_risk(metrics_payload=metrics, forecast_horizon_days=14)
    
    print('\n>>> Discord Embed Payload Preview:')
    print(f"• Title: 🔥 Wildfire Risk Assessment: {reg['name']}")
    print(f"• Vulnerability: {forecast['wildfire_vulnerability_index']} / 1.00 ({forecast['risk_tier']})")
    if reg['peatland_pct'] > 30:
        pdi = metrics['peatland_drying_index']
        print(f"• Peat Water Table: {pdi['estimated_water_table_cm']} cm (Status: {pdi['status']})")
    else:
        print('• Soil Classification: Non-Peat Volcanic Mineral Savanna (0% Peat)')
    print(f"• 14d Dry Spell: {forecast['weather_summary']['consecutive_dry_days']} consecutive rainless days")
    print(f"• Top Driver: {forecast['top_drivers'][0]['feature']} ({forecast['top_drivers'][0]['contribution_pct']}%)")
    print(f"• Advisory: {forecast['recommended_advisory']}")

# Run tests for both Volcanic Savanna (Bromo) and Peatland (Kapuas)
async def main():
    await test_risk_command('Mount Bromo National Park')
    await test_risk_command('Kapuas Regency')

await main() if 'get_ipython' in globals() else asyncio.run(main())

---
## 4. Command: `/hotspots [region] [lookback_days]`

### Technical Description
The `/hotspots` command queries near-real-time thermal anomaly detections from **NASA FIRMS** (VIIRS NOAA-20/21 375m and MODIS 1km) within the target jurisdiction.

### Key Features
- **Peatland & Conservation Boundary Intersection**: Automatically cross-references coordinates against the Ministry of Environment and Forestry (KLHK) Peatland Hydrological Units (*Kesatuan Hidrologi Gambut* / KHG) or National Park conservation zones.
- **Fire Radiative Power (FRP)**: Measures instantaneous thermal radiant energy in Megawatts (MW), signaling fire intensity.
- **Haze Dispersion Modeling**: Uses dominant wind vector $U/V$ components to project directional smoke cones downwind.

In [ ]:
# Executable Demonstration: Simulate /hotspots Command
from app.agent.tools import tool_fetch_active_hotspots

async def test_hotspots_command(region_query: str, days: int = 3):
    reg = resolve_region(region_query)
    data = await tool_fetch_active_hotspots(region_name=reg['name'], lookback_days=days)
    
    print(f'\n{"="*60}')
    print(f"Executing /hotspots region: '{reg['name']}' lookback_days: {days}")
    print(f"Total Detections: {data['total_hotspots']}")
    print(f"In Peat / Protected Zone: {data['peatland_hotspots']}")
    print(f"Total FRP: {data['total_frp_mw']} MW (Max: {data['max_frp_mw']} MW)")
    print('\nSample Active Thermal Coordinates:')
    for h in data['hotspots'][:3]:
        print(f"  • [{h['latitude']}, {h['longitude']}] FRP: {h['frp_mw']} MW ({h['confidence']} conf) — {h['khg_name']}")
    
    cones = data.get('haze_dispersion_cones', {}).get('features', [])
    print(f'Forward Smoke Plumes Computed: {len(cones)} directional cones')

await test_hotspots_command('Kapuas Regency', 3) if 'get_ipython' in globals() else asyncio.run(test_hotspots_command('Kapuas Regency', 3))

---
## 5. Command: `/ask [query]`

### Technical Description
The `/ask` command triggers the autonomous **ReAct (Reasoning + Acting) Agent Core**. The agent interprets arbitrary natural language questions, extracts geographic intent, executes necessary remote sensing tools sequentially, and formats an executive tactical briefing in Markdown.

In [ ]:
# Executable Demonstration: Simulate /ask Command with Autonomous ReAct Agent
from app.agent.engine import PyroGuardAgentEngine

async def test_ask_command(query_text: str):
    print(f'\n{"="*60}')
    print(f'Executing /ask query: "{query_text}"')
    
    engine = PyroGuardAgentEngine()
    result = await engine.execute_query(query=query_text)
    
    print('\n>>> Synthesized Report Output (Truncated Preview):')
    lines = result['report_markdown'].split('\n')
    for line in lines[:18]:
        print(line)

await test_ask_command('Assess current wildfire vulnerability at Mount Bromo National Park') if 'get_ipython' in globals() else asyncio.run(test_ask_command('Assess current wildfire vulnerability at Mount Bromo National Park'))

---
## 6. Command: `/regions`

### Technical Description
The `/regions` command lists all supported administrative territories, peat dome complexes (*kubah gambut*), and national parks registered in the PyroGuard AI database.

In [ ]:
# Executable Demonstration: Display Registered Jurisdictions
from app.domain.indonesia import REGIONS_REGISTRY

print(f'{"Region":<30} {"Province":<20} {"Peat (%)":<10} Center [Lon, Lat]')
print('-' * 75)
for k, r in REGIONS_REGISTRY.items():
    print(f"{r['name']:<30} {r['province']:<20} {r['peatland_pct']:<10} {r['center']}")

---
## 7. Troubleshooting & Operational Guardrails

### 1. `discord.errors.LoginFailure: Improper token has been passed (401)`
- **Cause**: An Application ID, Public Key, or OAuth2 Client Secret was placed in `DISCORD_BOT_TOKEN` instead of the Bot Token.
- **Solution**: Open the [Discord Developer Portal](https://discord.com/developers/applications) → Select your App → **Bot** (left tab) → Click **Reset Token** and copy the resulting `~70`-character token into `.env`.

### 2. Slash Commands Do Not Appear in Discord
- **Cause**: The bot was invited without the `applications.commands` OAuth2 scope, or command synchronization is pending.
- **Solution**: Re-invite the bot using the URL Generator with both `bot` and `applications.commands` checked. PyroGuard AI executes `tree.sync()` automatically upon startup.

### 3. Interaction Timeouts
- **Design Safeguard**: All PyroGuard AI Discord commands call `await interaction.response.defer(thinking=True)` upfront, extending Discord's default 3-second acknowledgement window to 15 minutes for remote sensing pipelines.